<a href="https://colab.research.google.com/github/MdShajalalsojib/Machine-Learning-Lab/blob/main/Lab_Report_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
sh = pd.read_csv('bangla_spam.csv')

In [3]:
sh.head()

,type,text
0,spam,এই মেসেজটি শেয়ার করুন এবং জিতে নিন আকর্ষণীয় ...
1,spam,আপনার বন্ধুদের রেফার করুন এবং প্রতি রেফারেলে ২...
2,ham,ট্রানজ্যাকশন নম্বর R234321.1554.640085 20 টাকা...
3,spam,নতুন অফার! সীমিত সময়ের জন্য পণ্যের উপর ৯০% পর...
4,ham,নববর্ষের শুভেচ্ছা!! আল্লাহ আপনার সকল কষ্ট দূর ...


In [7]:
sh.shape

(2602, 2)

In [8]:
import re

def clean_text(text):
    text = re.sub(r'[^\u0980-\u09FF\s]', '', str(text))
    return text

sh['clean_text'] = sh['text'].apply(clean_text)

sh['tokens'] = sh['clean_text'].apply(lambda x: x.split())

stopwords = ['আমি','তুমি','এই','ওই','সে','এটা']
sh['tokens'] = sh['tokens'].apply(lambda x: [word for word in x if word not in stopwords])

def stemming(words):
    return [word[:-1] if len(word) > 4 else word for word in words]

sh['final'] = sh['tokens'].apply(stemming)

sh['final_text'] = sh['final'].apply(lambda x: " ".join(x))

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(sh['final_text']).toarray()

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(sh['type'])

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()

model.add(Dense(64, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=5, batch_size=32)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7439 - loss: 0.6397
Epoch 2/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9250 - loss: 0.3079
Epoch 3/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9543 - loss: 0.1396
Epoch 4/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9688 - loss: 0.0907
Epoch 5/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9774 - loss: 0.0626


In [17]:
loss, accuracy = model.evaluate(X_test, y_test)
print("ANN Accuracy:", accuracy)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9424 - loss: 0.1641  
ANN Accuracy: 0.9424183964729309


In [18]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred = nb.predict(X_test)

nb_acc = accuracy_score(y_test, y_pred)
print("Naive Bayes Accuracy:", nb_acc)


Naive Bayes Accuracy: 0.9328214971209213


In [19]:
sh.head(20)

,type,text,clean_text,tokens,final,final_text
0,spam,এই মেসেজটি শেয়ার করুন এবং জিতে নিন আকর্ষণীয় ...,এই মেসেজটি শেয়ার করুন এবং জিতে নিন আকর্ষণীয় ...,"[মেসেজটি, শেয়ার, করুন, এবং, জিতে, নিন, আকর্ষণ...","[মেসেজট, শেয়া, করুন, এবং, জিতে, নিন, আকর্ষণীয...",মেসেজট শেয়া করুন এবং জিতে নিন আকর্ষণীয পুরস্কা
1,spam,আপনার বন্ধুদের রেফার করুন এবং প্রতি রেফারেলে ২...,আপনার বন্ধুদের রেফার করুন এবং প্রতি রেফারেলে ২...,"[আপনার, বন্ধুদের, রেফার, করুন, এবং, প্রতি, রেফ...","[আপনা, বন্ধুদে, রেফা, করুন, এবং, প্রত, রেফারেল...",আপনা বন্ধুদে রেফা করুন এবং প্রত রেফারেল ২০০ টা...
2,ham,ট্রানজ্যাকশন নম্বর R234321.1554.640085 20 টাকা...,ট্রানজ্যাকশন নম্বর টাকা রিচার্জটি সফল হয়েছে ...,"[ট্রানজ্যাকশন, নম্বর, টাকা, রিচার্জটি, সফল, হয়...","[ট্রানজ্যাকশ, নম্ব, টাকা, রিচার্জট, সফল, হয়েছ,...",ট্রানজ্যাকশ নম্ব টাকা রিচার্জট সফল হয়েছ আপনা ব...
3,spam,নতুন অফার! সীমিত সময়ের জন্য পণ্যের উপর ৯০% পর...,নতুন অফার সীমিত সময়ের জন্য পণ্যের উপর ৯০ পর্য...,"[নতুন, অফার, সীমিত, সময়ের, জন্য, পণ্যের, উপর,...","[নতুন, অফার, সীমি, সময়ে, জন্য, পণ্যে, উপর, ৯০...",নতুন অফার সীমি সময়ে জন্য পণ্যে উপর ৯০ পর্যন্ ...
4,ham,নববর্ষের শুভেচ্ছা!! আল্লাহ আপনার সকল কষ্ট দূর ...,নববর্ষের শুভেচ্ছা আল্লাহ আপনার সকল কষ্ট দূর কর...,"[নববর্ষের, শুভেচ্ছা, আল্লাহ, আপনার, সকল, কষ্ট,...","[নববর্ষে, শুভেচ্ছ, আল্লা, আপনা, সকল, কষ্ট, দূর...",নববর্ষে শুভেচ্ছ আল্লা আপনা সকল কষ্ট দূর করে আপ...
5,spam,আপনার ইমেইল যাচাই করুন এবং ২০০০ টাকা বোনাস জিতুন!,আপনার ইমেইল যাচাই করুন এবং ২০০০ টাকা বোনাস জিতুন,"[আপনার, ইমেইল, যাচাই, করুন, এবং, ২০০০, টাকা, ব...","[আপনা, ইমেই, যাচা, করুন, এবং, ২০০০, টাকা, বোনা...",আপনা ইমেই যাচা করুন এবং ২০০০ টাকা বোনা জিতু
6,spam,নতুন গ্যাজেট কিনুন এবং ১০% ক্যাশব্যাক পান!,নতুন গ্যাজেট কিনুন এবং ১০ ক্যাশব্যাক পান,"[নতুন, গ্যাজেট, কিনুন, এবং, ১০, ক্যাশব্যাক, পান]","[নতুন, গ্যাজে, কিনু, এবং, ১০, ক্যাশব্যা, পান]",নতুন গ্যাজে কিনু এবং ১০ ক্যাশব্যা পান
7,spam,আপনার মোবাইল রিচার্জ করুন এবং ১৫% ক্যাশব্যাক পান!,আপনার মোবাইল রিচার্জ করুন এবং ১৫ ক্যাশব্যাক পান,"[আপনার, মোবাইল, রিচার্জ, করুন, এবং, ১৫, ক্যাশব...","[আপনা, মোবাই, রিচার্, করুন, এবং, ১৫, ক্যাশব্যা...",আপনা মোবাই রিচার্ করুন এবং ১৫ ক্যাশব্যা পান
8,spam,বিশেষ ছাড়! সব পণ্যের উপর ৯৫% পর্যন্ত ডিসকাউন্ট!,বিশেষ ছাড় সব পণ্যের উপর ৯৫ পর্যন্ত ডিসকাউন্ট,"[বিশেষ, ছাড়, সব, পণ্যের, উপর, ৯৫, পর্যন্ত, ডি...","[বিশে, ছাড়, সব, পণ্যে, উপর, ৯৫, পর্যন্, ডিসকা...",বিশে ছাড় সব পণ্যে উপর ৯৫ পর্যন্ ডিসকাউন্
9,spam,আজই ক্লিক করুন এবং ৫০০০ টাকা জিতুন!,আজই ক্লিক করুন এবং ৫০০০ টাকা জিতুন,"[আজই, ক্লিক, করুন, এবং, ৫০০০, টাকা, জিতুন]","[আজই, ক্লি, করুন, এবং, ৫০০০, টাকা, জিতু]",আজই ক্লি করুন এবং ৫০০০ টাকা জিতু


In [21]:
sh.head()

,type,text,clean_text,tokens,final,final_text
0,spam,এই মেসেজটি শেয়ার করুন এবং জিতে নিন আকর্ষণীয় ...,এই মেসেজটি শেয়ার করুন এবং জিতে নিন আকর্ষণীয় ...,"[মেসেজটি, শেয়ার, করুন, এবং, জিতে, নিন, আকর্ষণ...","[মেসেজট, শেয়া, করুন, এবং, জিতে, নিন, আকর্ষণীয...",মেসেজট শেয়া করুন এবং জিতে নিন আকর্ষণীয পুরস্কা
1,spam,আপনার বন্ধুদের রেফার করুন এবং প্রতি রেফারেলে ২...,আপনার বন্ধুদের রেফার করুন এবং প্রতি রেফারেলে ২...,"[আপনার, বন্ধুদের, রেফার, করুন, এবং, প্রতি, রেফ...","[আপনা, বন্ধুদে, রেফা, করুন, এবং, প্রত, রেফারেল...",আপনা বন্ধুদে রেফা করুন এবং প্রত রেফারেল ২০০ টা...
2,ham,ট্রানজ্যাকশন নম্বর R234321.1554.640085 20 টাকা...,ট্রানজ্যাকশন নম্বর টাকা রিচার্জটি সফল হয়েছে ...,"[ট্রানজ্যাকশন, নম্বর, টাকা, রিচার্জটি, সফল, হয়...","[ট্রানজ্যাকশ, নম্ব, টাকা, রিচার্জট, সফল, হয়েছ,...",ট্রানজ্যাকশ নম্ব টাকা রিচার্জট সফল হয়েছ আপনা ব...
3,spam,নতুন অফার! সীমিত সময়ের জন্য পণ্যের উপর ৯০% পর...,নতুন অফার সীমিত সময়ের জন্য পণ্যের উপর ৯০ পর্য...,"[নতুন, অফার, সীমিত, সময়ের, জন্য, পণ্যের, উপর,...","[নতুন, অফার, সীমি, সময়ে, জন্য, পণ্যে, উপর, ৯০...",নতুন অফার সীমি সময়ে জন্য পণ্যে উপর ৯০ পর্যন্ ...
4,ham,নববর্ষের শুভেচ্ছা!! আল্লাহ আপনার সকল কষ্ট দূর ...,নববর্ষের শুভেচ্ছা আল্লাহ আপনার সকল কষ্ট দূর কর...,"[নববর্ষের, শুভেচ্ছা, আল্লাহ, আপনার, সকল, কষ্ট,...","[নববর্ষে, শুভেচ্ছ, আল্লা, আপনা, সকল, কষ্ট, দূর...",নববর্ষে শুভেচ্ছ আল্লা আপনা সকল কষ্ট দূর করে আপ...
